In [1]:
from pyspark.sql.functions import *
from travel_assistant import get_spark_session

In [ ]:
spark = get_spark_session()

In [3]:
COLUMNS = [
    "dt", "date", "hour", "zip_code", "address", "sale_type", "label", 
    "municipality", "province", "locality", "gasoline_95_e5_price"
]
df = spark.read.format("delta").load("../data/spain-fuel-price/")

In [4]:
df.count()

155282

In [5]:
df_counts = df.groupBy(df.columns).count()
duplicate_rows = df_counts.filter(col("count") > 1)
duplicate_rows.toPandas()

,dt,date,hour,zip_code,municipality_id,province_id,sale_type,label,address,municipality,...,diesel_a_price,diesel_b_price,diesel_premium_price,gasoline_95_e10_price,gasoline_95_e5_price,gasoline_95_e5_premium_price,gasoline_98_e10_price,gasoline_98_e5_price,hydrogen_price,count
0,2024-03-13T15:23:04+00:00,2024-03-13,15,8880,938,8,p,repsol,"carretera c-31 km. 148,50",cubelles,...,1.619,NaN,1.689,NaN,1.709,NaN,NaN,1.859,NaN,2
1,2024-03-14T15:14:41+00:00,2024-03-14,15,8880,938,8,p,repsol,"carretera c-31 km. 148,50",cubelles,...,1.599,NaN,1.669,NaN,1.719,NaN,NaN,1.869,NaN,2
2,2024-03-14T13:33:16+00:00,2024-03-14,13,8880,938,8,p,repsol,"carretera c-31 km. 148,50",cubelles,...,1.599,NaN,1.669,NaN,1.719,NaN,NaN,1.869,NaN,2


In [8]:
(df
 .select(COLUMNS)
 .where(col("zip_code").eqNullSafe("28050") & col("gasoline_95_e5_price").isNotNull())
 .groupBy("label", "address")
 .agg(
     count("address").alias("count"),
     min("gasoline_95_e5_price").alias("min"),
     mean("gasoline_95_e5_price").alias("mean"),
     max("gasoline_95_e5_price").alias("max"),
 )
 .show(100, truncate=False)
)

+------+---------------------------+-----+-----+------------------+-----+
|label |address                    |count|min  |mean              |max  |
+------+---------------------------+-----+-----+------------------+-----+
|repsol|calle abetal, 8            |13   |1.759|1.768230769230769 |1.779|
|cepsa |calle maria de portugal, 15|13   |1.707|1.7153846153846157|1.748|
|cepsa |avenida manoteras, 34      |13   |1.689|1.6997692307692307|1.729|
+------+---------------------------+-----+-----+------------------+-----+



In [7]:
(df
 .select("dt", "date", "hour", "address", "gasoline_95_e5_price")
 .where(col("address").eqNullSafe("calle maria de portugal, 15"))
 .orderBy("dt")
 .show(100, truncate=False)
)

+-------------------------+----------+----+---------------------------+--------------------+
|dt                       |date      |hour|address                    |gasoline_95_e5_price|
+-------------------------+----------+----+---------------------------+--------------------+
|2024-03-13T15:23:04+00:00|2024-03-13|15  |calle maria de portugal, 15|1.748               |
|2024-03-13T17:56:15+00:00|2024-03-13|17  |calle maria de portugal, 15|1.748               |
|2024-03-14T13:33:16+00:00|2024-03-14|13  |calle maria de portugal, 15|1.707               |
|2024-03-14T15:14:41+00:00|2024-03-14|15  |calle maria de portugal, 15|1.707               |
|2024-03-14T17:52:44+00:00|2024-03-14|17  |calle maria de portugal, 15|1.707               |
|2024-03-14T19:39:41+00:00|2024-03-14|19  |calle maria de portugal, 15|1.707               |
|2024-03-15T15:24:24+00:00|2024-03-15|15  |calle maria de portugal, 15|1.707               |
|2024-03-15T18:05:37+00:00|2024-03-15|18  |calle maria de portugal, 15

In [9]:
(df
 .select("dt", "date", "hour", "address", "gasoline_95_e5_price")
 .where(col("address").eqNullSafe("avenida manoteras, 34"))
 .orderBy("dt")
 .show(100, truncate=False)
)

+-------------------------+----------+----+---------------------+--------------------+
|dt                       |date      |hour|address              |gasoline_95_e5_price|
+-------------------------+----------+----+---------------------+--------------------+
|2024-03-13T15:23:04+00:00|2024-03-13|15  |avenida manoteras, 34|1.689               |
|2024-03-13T17:56:15+00:00|2024-03-13|17  |avenida manoteras, 34|1.689               |
|2024-03-14T13:33:16+00:00|2024-03-14|13  |avenida manoteras, 34|1.689               |
|2024-03-14T15:14:41+00:00|2024-03-14|15  |avenida manoteras, 34|1.689               |
|2024-03-14T17:52:44+00:00|2024-03-14|17  |avenida manoteras, 34|1.689               |
|2024-03-14T19:39:41+00:00|2024-03-14|19  |avenida manoteras, 34|1.689               |
|2024-03-15T15:24:24+00:00|2024-03-15|15  |avenida manoteras, 34|1.689               |
|2024-03-15T18:05:37+00:00|2024-03-15|18  |avenida manoteras, 34|1.689               |
|2024-03-16T09:48:16+00:00|2024-03-16|9   |